In [20]:
from pathlib import Path

import pandas as pd

from eutl_scraper import Settings
from eutl_scraper.eutl.extract.account_holders import (
    _load_accounts_from_manual_data,
)
from eutl_scraper.eutl.mappings import map_registryCode_inv

In [21]:
settings = Settings("data_tmp/")
fn_trans = settings.fp("transactions", settings.dir_source)
fn_manual = Path("manual_data/accounts_20260412.xlsx")

## Load data

### Transaction accounts

In [22]:
df_trans = pd.read_csv(fn_trans)

/var/folders/w2/k3rfwxh15jv4m5ttwmmc55j00000gn/T/ipykernel_79332/2553340399.py:1: DtypeWarning: Columns (0: TRANSFERRING_INSTALLATION_NAME, 1: TRANSFERRING_INSTALLATION_PERMIT_IDENTIFIER, 2: TRANSFERRING_INSTALLATION_PARENT_COMPANY, 3: TRANSFERRING_INSTALLATION_SUBSIDIARY_COMPANY, 4: TRANSFERRING_INSTALLATION_EPER_IDENTIFICATION, 5: TRANSFERRING_INSTALLATION_CITY, 6: TRANSFERRING_INSTALLATION_POSTAL_CODE, 7: TRANSFERRING_INSTALLATION_ADDRESS1, 8: TRANSFERRING_INSTALLATION_ADDRESS2, 9: ACQUIRING_INSTALLATION_NAME, 10: ACQUIRING_INSTALLATION_PERMIT_IDENTIFIER, 11: ACQUIRING_INSTALLATION_PARENT_COMPANY, 12: ACQUIRING_INSTALLATION_SUBSIDIARY_COMPANY, 13: ACQUIRING_INSTALLATION_EPER_IDENTIFICATION, 14: ACQUIRING_INSTALLATION_CITY, 15: ACQUIRING_INSTALLATION_POSTAL_CODE, 16: ACQUIRING_INSTALLATION_ADDRESS1, 17: ACQUIRING_INSTALLATION_ADDRESS2, 18: SUPP_UNIT_TYPE_DESCRIPTION, 19: LULUCF_CODE_DESCRIPTION, 20: EXPIRY_DATE) have mixed types. Specify dtype option on import or set low_memory=False

In [ ]:
def get_transaction_accounts(df: pd.DataFrame) -> pd.DataFrame:
    """Get transaction accounts from the transactions data.

    Args:
        df (pd.DataFrame): DataFrame with transactions data.

    Returns:
        pd.DataFrame: DataFrame with transaction accounts."""
    prefixes = ["TRANSFERRING", "ACQUIRING"]
    lst_df = []
    for prefix in prefixes:
        cols = {
            f"{prefix}_REGISTRY_NAME": "registry_name",
            f"{prefix}_ACCOUNT_TYPE1": "account_type1",
            f"{prefix}_ACCOUNT_TYPE2": "account_type2",
            f"{prefix}_ACCOUNT_TYPE3": "account_type3",
            f"{prefix}_ACCOUNT_OPEN_DT": "account_open_dt",
            f"{prefix}_ACCOUNT_END_OF_VALIDITY": "account_end_of_validity",
            f"{prefix}_ACCOUNT_NAME": "account_name",
            f"{prefix}_ACCOUNT_IDENTIFIER": "account_identifier",
            f"{prefix}_ACCOUNT_HOLDER": "account_holder_name",
            f"{prefix}_ACCOUNT_HOLDER_ADDRESS1": "account_holder_address1",
            f"{prefix}_ACCOUNT_HOLDER_ADDRESS2": "account_holder_address2",
            f"{prefix}_ACCOUNT_HOLDER_CITY": "account_holder_city",
            f"{prefix}_ACCOUNT_HOLDER_POSTAL_CODE": "account_holder_postal_code",
            f"{prefix}_ACCOUNT_HOLDER_COUNTRY_CODE": "account_holder_country_code",
            f"{prefix}_ACCOUNT_HOLDER_COMPANY_REGISTRATION_NUMBER": "account_holder_company_registration_number",
            f"{prefix}_ACCOUNT_HOLDER_LEI": "account_holder_lei",
        }
        cols = {
            "Account Identifier": "account_identifier",
            "National Administrator": "registry_name",
            "Account Type": "account_type",
            "Account Holder Name": "account_holder_name",
            "Account Name": "account_name",
            # "Installation/Aircraft Operator/Maritime Operator ID": "installation_aircraft_operator_maritime_operator_id",
            "Company Registration No": "company_registration_no",
            "Main Address Line": "main_address_line",
            "City": "city",
            "Legal Entity Identifier": "legal_entity_identifier",
            # "Telephone 1": "telephone_1",
            # "Telephone 2": "telephone_2",
            "Email": "email",
            "registry_id": "registry_id",
            "account_id": "account_id",
        }
        df_ = (
            df[list(cols.keys())]
            .rename(columns=cols)
            .assign(
                registry_id=lambda x: x["registry_name"]
                .str.strip()
                .map(map_registryCode_inv),
            )
        )
        lst_df.append(df_)
    df_trans_accounts = (
        pd.concat(lst_df, ignore_index=True)
        .dropna(subset=["account_identifier"])
        .assign(
            account_id=lambda df: (
                df["registry_id"] + "_" + df["account_identifier"].astype(str)
            )
        )
        .drop_duplicates()
    )
    assert df_trans_accounts.account_id.is_unique, "account_id is not unique"
    return df_trans_accounts


df_trans_accounts = get_transaction_accounts(df_trans)

### Manual accounts

In [25]:
df_manual = _load_accounts_from_manual_data(fn_manual)

for c in df_manual.columns:
    print(f'"{c}": "{c}",')

"Account Identifier": "Account Identifier",
"National Administrator": "National Administrator",
"Account Type": "Account Type",
"Account Holder Name": "Account Holder Name",
"Account Name": "Account Name",
"Installation/Aircraft Operator/Maritime Operator ID": "Installation/Aircraft Operator/Maritime Operator ID",
"Company Registration No": "Company Registration No",
"Main Address Line": "Main Address Line",
"City": "City",
"Legal Entity Identifier": "Legal Entity Identifier",
"Telephone 1": "Telephone 1",
"Telephone 2": "Telephone 2",
"Email": "Email",
"registry_id": "registry_id",
"account_id": "account_id",


In [ ]:
cols = {
    "Account Identifier": "account_identifier",
    "National Administrator": "national_administrator",
    "Account Type": "account_type",
    "Account Holder Name": "account_holder_name",
    "Account Name": "account_name",
    # "Installation/Aircraft Operator/Maritime Operator ID": "installation_aircraft_operator_maritime_operator_id",
    "Company Registration No": "company_registration_no",
    "Main Address Line": "main_address_line",
    "City": "city",
    "Legal Entity Identifier": "legal_entity_identifier",
    "Telephone 1": "telephone_1",
    "Telephone 2": "telephone_2",
    "Email": "email",
    "registry_id": "registry_id",
    "account_id": "account_id",
}